In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import TargetEncoder
from sklearn.preprocessing import RobustScaler
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

from xgboost import XGBClassifier

In [ ]:
Churn_client= pd.read_csv("Dataset_churn.csv")
Churn_client

,msisdn,tenure_mths_cnt,gadget_type,device_category,smart_phone_flag,manufacturer,active_days_count_mth2,active_days_count_mth3,active_days_count_mth4,data_active_days_count_mth2,...,avg_paid_data_usage,avg_voice_usage,avg_voice_onnet_usage,avg_voice_offnet_usage,avg_recharge_tot_amount,max_recharge_tot_amount,active_days_count_diff_mth2_3,active_days_count_diff_mth3_4,avg_active_days_diff,value_segment
0,028D60F7F40E18BC63C43883B6B6EFD7,0.000000,unknown,Unknown,Unknown,unknown,0.000000,0.131148,0.032258,0.000000,...,3.392080e-04,0.001130,0.001445,0.000000,0.000170,0.000411,0.733333,0.406593,0.483871,Medium
1,4AB05A575719E7BFC3DD30968D608367,0.405738,mobile handset,2G,No,itel,0.129032,0.180328,0.032258,0.096774,...,0.000000e+00,0.000970,0.001241,0.000000,0.000158,0.000316,0.766667,0.439560,0.548387,Low
2,47366A6051322F08A46B327FBBAD03AC,0.086066,mobile handset,2G,No,itel,0.032258,0.081967,0.129032,0.032258,...,0.000000e+00,0.000749,0.000745,0.000503,0.000035,0.000047,0.866667,0.340659,0.451613,Low
3,B7221E307F39F210781D6E9529B213D1,0.057377,unknown,Unknown,Unknown,unknown,0.000000,0.049180,0.129032,0.032258,...,6.340000e-07,0.000004,0.000001,0.000009,0.000000,0.000000,0.900000,0.318681,0.435484,Low
4,A6086B2E16BF1630AD75AEA6D21714C2,0.040984,mobile handset,2G,No,tecno,0.774194,0.721311,0.032258,0.032258,...,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.333333,0.802198,0.870968,Low
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91359,654DEFDC7578BA01E387D0E6DDE54B34,0.221311,mobile handset,2G,No,itel,0.580645,0.655738,0.806452,0.000000,...,0.000000e+00,0.000539,0.000690,0.000000,0.000111,0.000126,0.266667,0.494505,0.387097,Low
91360,B71BCFA51DF724AF26608EBEA73F478C,0.131148,mobile handset,2G,No,other,0.096774,0.131148,0.258065,0.000000,...,0.000000e+00,0.002026,0.002060,0.001257,0.000192,0.000237,0.833333,0.329670,0.419355,Low
91361,F11DF3E6E69D487A8787701F56AC0CC4,0.004098,mobile handset,Non-Data,No,itel,0.870968,0.655738,0.451613,0.032258,...,0.000000e+00,0.002060,0.002635,0.000000,0.000697,0.001170,0.566667,0.615385,0.709677,Low
91362,DDABC68BFDFB39F967ECE77F7BA291AD,0.024590,mobile handset,Non-Data,No,samsung,0.129032,0.229508,0.193548,0.000000,...,0.000000e+00,0.000822,0.000051,0.002364,0.000216,0.000364,0.666667,0.417582,0.467742,Low


In [ ]:
# 1. Remplacer les 'unknown' par NaN
# 1. Remplacer les 'unknown' par NaN
df_str = Churn_client.astype(str).apply(lambda x: x.str.lower())
masque_unknown = df_str.apply(lambda x: x.str.contains('unknow', regex=False))

for col in Churn_client.columns:
    if col in masque_unknown.columns:
        Churn_client.loc[masque_unknown[col], col] = np.nan

# 2. Convertir et nettoyer les annotations scientifiques -> NaN
pattern_scientifique = r'^[+-]?\d+(\.\d+)?[eE][+-]?\d+$'
masque_sci = Churn_client.astype(str).apply(lambda x: x.str.match(pattern_scientifique))
cols_sci = masque_sci.sum()[masque_sci.sum() > 0].index.tolist()

for col in cols_sci:
    Churn_client[col] = pd.to_numeric(Churn_client[col], errors='coerce')

# 3. Imputation par le MODE pour les colonnes contenant initialement des 'unknown' (variables catégorielles)
cols_unknown = masque_unknown.sum()[masque_unknown.sum() > 0].index.tolist()
cols_categoriques = ['gadget_type', 'device_category', 'smart_phone_flag', 'manufacturer']

for col in cols_categoriques:
    # Calcul du mode en ignorant les NaN
    mode_val = Churn_client[col].dropna().mode()[0]
    # Remplacement des NaN par la valeur du mode
    Churn_client[col] = Churn_client[col].fillna(mode_val)
    
# 4. Imputation par la MÉDIANE pour les colonnes contenant des annotations scientifiques (variables numériques)
for col in cols_sci:
    if Churn_client[col].isnull().sum() > 0:
        mediane_val = Churn_client[col].median()
        Churn_client[col] = Churn_client[col].fillna(mediane_val)

print("Imputation terminée avec succès !")
print(f"NaN restants dans les colonnes à 'unknown' : {Churn_client[cols_unknown].isnull().sum().sum()}")

print(f"NaN restants dans les colonnes à annotations scientifiques : {Churn_client[cols_sci].isnull().sum().sum()}")

Imputation terminée avec succès !
NaN restants dans les colonnes à 'unknown' : 0
NaN restants dans les colonnes à annotations scientifiques : 0
